# Pretrain TCR chain encoders on the gathered corpus

Self-supervised per-chain **autoencoder** warm-start (Vα, Vβ, CDR3α, CDR3β) on the
deduplicated multi-database corpus in `data/tcr_pretraining/tcr_pretrain_corpus.csv`
(~177k unique chains). Reconstruction is dominated by the germline scaffold; this is a
**warm-start / regularizer**, not a specificity learner.

Clones the repo into `/content`, writes outputs into the repo, then commits & pushes.
Run on Colab (GPU).

## Clone repo
Add your token in Colab **Secrets** (🔑 left sidebar) as `GITHUB_TOKEN`, then run.

In [ ]:
import os, sys, subprocess
REPO = '/content/tcrpmhc_pose_binding'
REPO_URL = 'github.com/92kunheekim/tcrpmhc_pose_binding.git'
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', f'https://x-access-token:{TOKEN}@{REPO_URL}', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, f'{REPO}/src')
DATA_DIR = f'{REPO}/data'
PRETRAIN_DIR = f'{REPO}/pretrained'; os.makedirs(PRETRAIN_DIR, exist_ok=True)
CORPUS = f'{DATA_DIR}/tcr_pretraining/tcr_pretrain_corpus.csv'
print('repo:', os.path.isdir(REPO), '| corpus:', os.path.isfile(CORPUS), '| out:', PRETRAIN_DIR)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'])
import pandas as pd, torch, matplotlib.pyplot as plt
from train_utils import pretrain_tcr_encoders, DEVICE
print('device:', DEVICE)

## Config

In [ ]:
TCR_EPOCHS = 40    # upper bound; plateau early-stop usually ends sooner
PATIENCE   = 5
BATCH      = 256

## 1. Load corpus

Columns `va, vb, cdr3a, cdr3b` are independently deduplicated and `""`-padded — 
`pretrain_tcr_encoders` drops the empty/padding entries per chain automatically.

In [ ]:
seq = pd.read_csv(CORPUS).fillna('')
print('rows:', len(seq))
print('non-empty per chain:', {c: int(seq[c].astype(str).str.strip().ne('').sum()) for c in ['va','vb','cdr3a','cdr3b']})

## 2. Pretrain the four chain autoencoders

In [ ]:
TCR_ENC, TCR_HIST = pretrain_tcr_encoders(
    seq, epochs=TCR_EPOCHS, bs=BATCH, patience=PATIENCE, return_history=True, log=True)
torch.save(TCR_ENC, f'{PRETRAIN_DIR}/tcr_encoders_corpus.pt')
pd.concat(TCR_HIST.values()).to_csv(f'{PRETRAIN_DIR}/tcr_encoder_corpus_history.csv', index=False)
print('saved -> tcr_encoders_corpus.pt')

## 3. Diagnostics: reconstruction MSE, amino-acid accuracy, active embedding dims

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for ch, h in TCR_HIST.items():
    ax[0].plot(h.epoch, h.mse, label=ch)
    ax[1].plot(h.epoch, h.aa_acc, label=ch)
    ax[2].plot(h.epoch, h.active_emb, label=ch)
ax[0].set(title='reconstruction MSE', xlabel='epoch')
ax[1].set(title='amino-acid recon accuracy', xlabel='epoch', ylim=(0, 1.01))
ax[2].set(title='active embedding dims', xlabel='epoch')
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{PRETRAIN_DIR}/tcr_encoder_corpus_curves.png', dpi=150); plt.show()
for ch, h in TCR_HIST.items():
    print(f'{ch:5s}: final mse={h.mse.values[-1]:.4f}  aa_acc={h.aa_acc.values[-1]:.3f}  active={int(h.active_emb.values[-1])}/{int(h.emb_dim.values[-1])}')

## 4. Commit & push outputs

In [ ]:
subprocess.run(['git', 'config', 'user.email', '92kunheekim@gmail.com'], cwd=REPO)
subprocess.run(['git', 'config', 'user.name', 'KH Kim'], cwd=REPO)
subprocess.run(['git', 'add', '-f', 'pretrained'], cwd=REPO, check=True)   # -f: .pt is gitignored
subprocess.run(['git', 'commit', '-m', 'Pretrained TCR chain encoders on corpus [colab]'], cwd=REPO)
subprocess.run(['git', 'push', 'origin', 'master'], cwd=REPO, check=True)
print('pushed.')

## Use in downstream training
```python
import torch
from train_utils import make_warm_start
TCR_ENC = torch.load(f'{PRETRAIN_DIR}/tcr_encoders_corpus.pt')
warm = make_warm_start(TCR_ENC)   # + pep_state / mhc_state / cpose_state if available
```